In [ ]:
!pip install pandas sentence-transformers scikit-learn

In [7]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------
# 1. LOAD & CLEAN DATA
# -----------------------------
def load_data(path="phd_studentships.csv"):

    df = pd.read_csv(path)

    # normalize column names
    df.columns = df.columns.str.strip().str.lower()

    required_cols = [
        "title",
        "employer",
        "department",
        "salary",
        "location",
        "url",
    ]

    df = df[required_cols]

    # drop rows without title
    df = df.dropna(subset=["title"])

    df = df.reset_index(drop=True)

    return df


# -----------------------------
# 2. BUILD TEXT FOR EMBEDDING
# -----------------------------
def build_corpus(df):
    """
    Combine useful text fields into one description
    for semantic similarity.
    """
    corpus = (
        df["title"].fillna("") + " "
        + df["department"].fillna("") + " "
        + df["employer"].fillna("")
    )
    return corpus.tolist()


# -----------------------------
# 3. CREATE EMBEDDINGS (BERT)
# -----------------------------
def create_embeddings(corpus):

    print("Loading Sentence-BERT model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    print("Encoding PhD descriptions...")
    embeddings = model.encode(corpus, show_progress_bar=True)

    return model, embeddings


# -----------------------------
# 4. RECOMMENDATION FUNCTION
# -----------------------------
def recommend_phds(query, model, embeddings, df, top_k=5):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(query_embedding, embeddings)[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results


# -----------------------------
# 5. MAIN PROGRAM
# -----------------------------
def main():

    print("=== AI PhD Opportunity Recommender ===\n")

    # Load dataset
    df = load_data("data.csv")

    print(f"Loaded {len(df)} PhD opportunities.\n")

    # Build corpus
    corpus = build_corpus(df)

    # Create embeddings
    model, embeddings = create_embeddings(corpus)

    # User query
    query = input("\nEnter your research interest: ")

    # Get recommendations
    results = recommend_phds(query, model, embeddings, df)

    # Display
    print("\n🎓 Top Recommended PhD Studentships:\n")

    for i, row in results.iterrows():
        print(f"Title: {row['title']}")
        print(f"University: {row['employer']}")
        print(f"Department: {row['department']}")
        print(f"Location: {row['location']}")
        print(f"Salary: {row['salary']}")
        print(f"Link: {row['url']}")
        print(f"Match Score: {row['similarity']:.3f}")
        print("-" * 60)


if __name__ == "__main__":
    main()


=== AI PhD Opportunity Recommender ===

Loaded 264 PhD opportunities.

Loading Sentence-BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding PhD descriptions...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


🎓 Top Recommended PhD Studentships:

Title: PhD Studentship: Large scale robust quantum computation with superconducting qubits
University: University of Surrey
Department: Physics
Location: Guildford
Salary: From £18,622 stipend amount, per annum, fees covered, research training support grants £3000. Funding is for 3.5 years
Link: https://www.jobs.ac.uk/job/DCF893/phd-studentship-large-scale-robust-quantum-computation-with-superconducting-qubits
Match Score: 0.539
------------------------------------------------------------
Title: PhD Studentship in Quantum Computing
University: The University of Edinburgh
Department: School of Informatics
Location: Edinburgh
Salary: Tax free stipend of £18,622 per year
Link: https://www.jobs.ac.uk/job/DCB754/phd-studentship-in-quantum-computing
Match Score: 0.531
------------------------------------------------------------
Title: PhD Studentship: Hybrid quantum communication and sensing networks
University: University of Sussex
Department: Physics a